<a href="https://colab.research.google.com/github/Dhavalkumar510/Fortray-Project/blob/main/WorkShop_For_Fortray.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Work Shop for Fortray

### Importing necessary libraries

In [1]:
import pandas as pd
import numpy as np

## Step 1: Robust Data Ingestion & Transformation

In [13]:
df = pd.read_csv("/content/hm_land_registry_2000.csv")

In [14]:
# Converting Transaction_Date to datetime as per requirement
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'],
                                        format='%d/%m/%Y', errors='coerce')

# Converting pricing columns to numeric
df['Asking_Price'] = pd.to_numeric(df['Asking_Price'], errors='coerce')
df['Sold_Price'] = pd.to_numeric(df['Sold_Price'], errors='coerce')
df['Days_on_Market'] = pd.to_numeric(df['Days_on_Market'], errors='coerce')

# Removing invalid rows:
df = df.dropna(subset=['Transaction_Date', 'Asking_Price', 'Sold_Price',
    'Days_on_Market'])

print(df.dtypes)
print("\n")
print("-"*85)
print("\n")
print(df.head())

Transaction_Date    datetime64[ns]
Region                      object
Property_Type               object
EPC_Rating                  object
Days_on_Market               int64
Asking_Price                 int64
Sold_Price                   int64
Market_Status               object
dtype: object


-------------------------------------------------------------------------------------


  Transaction_Date      Region  Property_Type EPC_Rating  Days_on_Market  \
0       2025-01-30      London       Detached          D             107   
1       2025-02-08  North West           Flat          A              97   
2       2025-03-29      London  Semi-Detached          C              19   
3       2025-01-17  South East       Detached          A             111   
4       2025-03-13    Scotland  Semi-Detached          F              76   

   Asking_Price  Sold_Price Market_Status  
0       1021987      981706        Normal  
1        163946       16395        Normal  
2        426746      427734

## Step 2: Heuristic Anomaly Filtering (The "Intelligent Filter")

In [8]:
# Detecting suspicious records
anomaly_mask = (
    (df['Sold_Price'] < 0.20 * df['Asking_Price']) |
    (df['Days_on_Market'] < 0)
)

# Separating anomalous and valid records as cleaned df files
quarantine_df = df[anomaly_mask].copy()
clean_df = df[~anomaly_mask].copy()

# quarantined records
quarantine_df.to_csv('quarantined_transactions.csv', index=False)

print(f"Quarantined {len(quarantine_df)} anomalous records.")
print(f"Retained {len(clean_df)} clean records.")

Quarantined 100 anomalous records.
Retained 1900 clean records.


##  Step 3: Standardization & Status Recalculation

In [9]:
# Recalculating Market_Status based on Days_on_Market
clean_df.loc[clean_df['Days_on_Market'] > 120, 'Market_Status'] = 'Stagnant'

clean_df.loc[clean_df['Days_on_Market'] < 14, 'Market_Status'] = 'High Demand'

clean_df.loc[
    (clean_df['Days_on_Market'] >= 14) &
    (clean_df['Days_on_Market'] <= 120),
    'Market_Status'
] = 'Normal'

print("Market_Status column successfully audited and corrected.")

Market_Status column successfully audited and corrected.


## Step 4: Algorithmic Feature Engineering

In [15]:
# Creating Price_Variance_Percent
clean_df['Price_Variance_Percent'] = (
    (clean_df['Sold_Price'] - clean_df['Asking_Price'])
    / clean_df['Asking_Price']
) * 100

# ingalculate regional quartiles
regional_quantiles = (
    clean_df.groupby('Region')['Sold_Price']
    .quantile([0.25, 0.50, 0.75])
    .unstack()
)

# Creating Valuation_Band
clean_df['Valuation_Band'] = 'Ultra-Premium'

for region in regional_quantiles.index:

    q1 = regional_quantiles.loc[region, 0.25]
    q2 = regional_quantiles.loc[region, 0.50]
    q3 = regional_quantiles.loc[region, 0.75]

    region_mask = clean_df['Region'] == region

    clean_df.loc[
        region_mask & (clean_df['Sold_Price'] <= q1),
        'Valuation_Band'
    ] = 'Entry-Level'

    clean_df.loc[
        region_mask &
        (clean_df['Sold_Price'] > q1) &
        (clean_df['Sold_Price'] <= q2),
        'Valuation_Band'
    ] = 'Mid-Market'

    clean_df.loc[
        region_mask &
        (clean_df['Sold_Price'] > q2) &
        (clean_df['Sold_Price'] <= q3),
        'Valuation_Band'
    ] = 'Premium'

print("Feature engineering completed successfully.")

# Displaying sample output as cleaned_df
print("\nProcessed Data Preview:")
print(clean_df.head())

# Save final processed dataset
clean_df.to_csv('processed_transactions.csv', index=False)

print("\nProcessed dataset saved as 'processed_transactions.csv'")

Feature engineering completed successfully.

Processed Data Preview:
  Transaction_Date      Region  Property_Type EPC_Rating  Days_on_Market  \
0       2025-01-30      London       Detached          D             107   
2       2025-03-29      London  Semi-Detached          C              19   
3       2025-01-17  South East       Detached          A             111   
5       2025-03-13      London  Semi-Detached          B              25   
6       2025-01-19   Yorkshire       Terraced          F             107   

   Asking_Price  Sold_Price Market_Status  Price_Variance_Percent  \
0       1021987      981706        Normal               -3.941440   
2        426746      427734        Normal                0.231519   
3        487823      497564        Normal                1.996831   
5        488570      491530        Normal                0.605850   
6        255896      267223        Normal                4.426408   

  Valuation_Band  
0  Ultra-Premium  
2     Mid-Market  
3 